In [ ]:
import numpy as np
import pandas as pd
import os
import copy

In [ ]:
INTERATION_INTERUPT = 10000

def fct_firstNoneSpaceCharOfLine(inpLine):

  for elem in inpLine:
    if elem != " ": return elem

  return " "

def fct_stripMarkdownTableLine(inpLine):
  while inpLine[ 0] != "|": inpLine = inpLine[1:]
  while inpLine[-1] != "|": inpLine = inpLine[:-1]

  inpLine = inpLine[1:-1]
  inpArgs = inpLine.split("|")

  for idxArg, inpArg in enumerate(inpArgs):
    inpArgs[idxArg] = inpArg.strip()

  return inpArgs

def fct_mdTableToDataframe(inpLines):

  # strip all lines before table starts

  it = 0

  while fct_firstNoneSpaceCharOfLine(inpLines[0]) != "|" and (it < INTERATION_INTERUPT):
    inpLines = inpLines[0:]
    it += 1

  # strip all lines after table

  it = 0

  while fct_firstNoneSpaceCharOfLine(inpLines[-1]) != "|" and (it < INTERATION_INTERUPT):
    inpLines = inpLines[:-1]
    it += 1

  header  = fct_stripMarkdownTableLine(inpLines[0])
  entries = []

  for elem in inpLines[2:]:
    entries.append(fct_stripMarkdownTableLine(elem))

  return pd.DataFrame(entries, columns=header)

In [ ]:
# load transactions

with open("pfioTransactionInput.md", "r") as fid:
  inpTracts = fid.readlines()

rawTracts  = fct_mdTableToDataframe(inpTracts)
wknsTracts = rawTracts["WKN"].unique()
wknsTracts = [wkn for wkn in wknsTracts if len(wkn) == 6]

In [ ]:
# calculate ivst budget

pos = rawTracts["Pos"]
sum = 0

for elem in pos: sum += int(elem.replace(".", ""))

print("Info ... IVST-Budget: %.2f" % (sum / 100))

In [ ]:
# read pfio param

with open("pfioParam.md", "r") as fid:
  pfioParam = fid.readlines()

pfioParam = fct_mdTableToDataframe(pfioParam)
wknParams = [elem for elem in pfioParam["WKN"]]

#          Name   Kurz Symbol    WKN Branche Region Typ
#1 Novo Nordisk NOVNOR  NOV.F A3EU6F    GUPH   EUGB   A

In [ ]:
# check wkns of transactions and parameters against each other

for wkn in wknsTracts:
  if wkn not in wknParams: print("Fehler: %s fehlt in den Portfolio Parametern!" % wkn)

for wkn in wknParams:
  if wkn not in wknsTracts: print("Warnung: %s taucht in keiner Transaktion auf!" % wkn)

In [ ]:
# calculate amount of share per title

pfio      = copy.deepcopy(pfioParam)
pfio      = pfio.set_index("WKN")
pfio["N"] = 0

for wkn in wknParams:
  titleTransactions = rawTracts[rawTracts["WKN"] == wkn]
  amountOfShare     = 0

  for index, row in titleTransactions.iterrows():
    if row["Typ"] == "EK": amountOfShare += int(row["N"].replace(".",""))
    if row["Typ"] == "VK": amountOfShare -= int(row["N"].replace(".",""))

  pfio.loc[wkn, "N"] = amountOfShare

In [ ]:
dataPathTracts = "./261010_tbData/"

os.makedirs(dataPathTracts, exist_ok=True)

for wkn in wkns:
  rawTractsTmp = rawTracts[rawTracts["WKN"] == wkn]

  rawTractsTmp.to_csv(dataPathTracts + wkn + ".csv", index=False)

  with open(dataPathTracts + wkn + ".dat", "w") as fid:
    fid.write(rawTractsTmp.to_string(index=False))
  